## Setup

This notebook requires Python 3.11 and the packages below. Run this command in a notebook cell once if they are not already installed:

```python
%pip install numpy==2.2.6 pandas==2.3.3
```

Restart the kernel after installation, then run the notebook from top to bottom.

In [1]:
#| label: setup
#| echo: true

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

## Learning goals

By the end of this introduction, you should be able to:

- distinguish an original table from a complete metric-ready representation;
- construct and interpret the missingness masks $M_X$ and $M_Y$;
- explain why one shared transformation must be used for $X$ and $Y$;
- follow a missing numerical or categorical value into $\widetilde X$ or $\widetilde Y$; and
- identify which future metrics inspect the original tables and which use complete representations.

## 1. Two tables with one schema

Let

$$
X=(x_{ij}),\qquad i=1,\ldots,n_X,\quad j=1,\ldots,p,
$$

and

$$
Y=(y_{kj}),\qquad k=1,\ldots,n_Y,\quad j=1,\ldots,p.
$$

The tables share the same $p$ columns and the same datatype for each column. They do **not** need the same number of rows, and their values may differ. For column $j$ with value space $\mathcal X_j$,

$$
x_{ij},y_{kj}\in\mathcal X_j\cup\{\mathrm{NA}\}.
$$

The symbol $\mathrm{NA}$ denotes a missing value. The names $X$ and $Y$ are deliberately neutral: either table could be observed, synthetic, reconstructed, or produced by another process.

In [2]:
#| label: construct-small-tables

X = pd.DataFrame(
    {
        "age": pd.Series([24, 31, pd.NA, 46, 58, 39], dtype="Int64"),
        "income_k": pd.Series([32, 48, 55, pd.NA, 91, 67], dtype="Float64"),
        "region": pd.Series(
            ["North", "South", "North", "West", pd.NA, "South"],
            dtype="string",
        ),
        "outcome": pd.Series([0, 0, 1, 1, 1, 0], dtype="Int64"),
    }
)
X.index = [f"x{i}" for i in range(1, len(X) + 1)]

Y = pd.DataFrame(
    {
        "age": pd.Series([25, pd.NA, 44, 60, 37], dtype="Int64"),
        "income_k": pd.Series([35, 51, pd.NA, 88, 64], dtype="Float64"),
        "region": pd.Series(
            ["North", "South", "West", pd.NA, "East"],
            dtype="string",
        ),
        "outcome": pd.Series([0, 1, 1, 1, 0], dtype="Int64"),
    }
)
Y.index = [f"y{i}" for i in range(1, len(Y) + 1)]

print(f"X has {len(X)} rows and {X.shape[1]} columns")
display(X)
print(f"Y has {len(Y)} rows and {Y.shape[1]} columns")
display(Y)

X has 6 rows and 4 columns


,age,income_k,region,outcome
x1,24,32.000,North,0
x2,31,48.000,South,0
x3,<NA>,55.000,North,1
x4,46,<NA>,West,1
x5,58,91.000,<NA>,1
x6,39,67.000,South,0


Y has 5 rows and 4 columns


,age,income_k,region,outcome
y1,25,35.000,North,0
y2,<NA>,51.000,South,1
y3,44,<NA>,West,1
y4,60,88.000,<NA>,1
y5,37,64.000,East,0


The example is intentionally small enough to inspect by eye:

- `age` and `income_k` are numerical;
- `region` is categorical;
- `outcome` is binary and will later play the role of a prediction target;
- both tables have missing values, but not in the same cells; and
- `East` occurs only in $Y$, illustrating why category definitions must be shared.

In [3]:
#| label: verify-shared-schema

schema = pd.DataFrame(
    {
        "X dtype": X.dtypes.astype(str),
        "Y dtype": Y.dtypes.astype(str),
        "same dtype?": X.dtypes.astype(str).eq(Y.dtypes.astype(str)),
    }
)
display(schema)
assert X.columns.equals(Y.columns)
assert X.dtypes.astype(str).equals(Y.dtypes.astype(str))

,X dtype,Y dtype,same dtype?
age,Int64,Int64,True
income_k,Float64,Float64,True
region,string,string,True
outcome,Int64,Int64,True


## 2. Missingness masks $M_X$ and $M_Y$

The missingness masks record **where** values are absent without filling them. Define

$$
(M_X)_{ij}=\mathbf 1\{x_{ij}=\mathrm{NA}\}
=
\begin{cases}
1, & x_{ij}\text{ is missing},\\
0, & x_{ij}\text{ is observed},
\end{cases}
$$

and define $M_Y$ in exactly the same way. Thus, $M_X$ has the same dimensions as $X$, while $M_Y$ has the same dimensions as $Y$.

In [4]:
#| label: missingness-masks

M_X = X.isna().astype("int8")
M_Y = Y.isna().astype("int8")

print("M_X: 1 means the corresponding cell in X is missing")
display(M_X)
print("M_Y: 1 means the corresponding cell in Y is missing")
display(M_Y)

M_X: 1 means the corresponding cell in X is missing


,age,income_k,region,outcome
x1,0,0,0,0
x2,0,0,0,0
x3,1,0,0,0
x4,0,1,0,0
x5,0,0,1,0
x6,0,0,0,0


M_Y: 1 means the corresponding cell in Y is missing


,age,income_k,region,outcome
y1,0,0,0,0
y2,1,0,0,0
y3,0,1,0,0
y4,0,0,1,0
y5,0,0,0,0


In [5]:
#| label: inspect-mask-cells

mask_examples = pd.DataFrame(
    [
        {
            "cell": "X[x3, age]",
            "original value": X.loc["x3", "age"],
            "mask value": M_X.loc["x3", "age"],
        },
        {
            "cell": "X[x4, age]",
            "original value": X.loc["x4", "age"],
            "mask value": M_X.loc["x4", "age"],
        },
        {
            "cell": "Y[y4, region]",
            "original value": Y.loc["y4", "region"],
            "mask value": M_Y.loc["y4", "region"],
        },
    ]
)
display(mask_examples)

,cell,original value,mask value
0,"X[x3, age]",<NA>,1
1,"X[x4, age]",46,0
2,"Y[y4, region]",<NA>,1


A natively complete table has a mask containing only zeros. For incomplete tables, the masks let us compare missingness without confusing an absent value with a numerical zero or an ordinary category.

## 3. One shared transformation

Many value-, utility-, and privacy-based metrics need complete numerical inputs. We therefore fit one transformation on the pooled pair $X\cup Y$ and apply the **same fitted rules** to both tables:

$$
\widetilde X=T_{X\cup Y}(X),
\qquad
\widetilde Y=T_{X\cup Y}(Y).
$$

For this introductory example, $T_{X\cup Y}$ does four transparent things:

1. learns one pooled median and range for each numerical column;
2. fills a missing numerical value with that pooled median and retains a missingness-indicator column;
3. learns one pooled codebook for categorical levels, including an explicit missing level; and
4. leaves the complete binary outcome unchanged.

Fitting unrelated transformations to $X$ and $Y$ could assign different scales or category codes to the same value, making the comparison meaningless.

In [6]:
#| label: fit-shared-transformation

NUMERIC_COLUMNS = ["age", "income_k"]
CATEGORICAL_COLUMN = "region"
MISSING_LEVEL = "__MISSING__"

pooled = pd.concat([X, Y], axis=0)
numeric_parameters = pd.DataFrame(
    {
        "median": pooled[NUMERIC_COLUMNS].median(),
        "minimum": pooled[NUMERIC_COLUMNS].min(),
        "maximum": pooled[NUMERIC_COLUMNS].max(),
    }
)
numeric_parameters["range"] = (
    numeric_parameters["maximum"] - numeric_parameters["minimum"]
)

observed_regions = sorted(pooled[CATEGORICAL_COLUMN].dropna().unique())
region_levels = [*observed_regions, MISSING_LEVEL]
region_to_code = {level: code for code, level in enumerate(region_levels)}
region_codebook = pd.DataFrame(
    {"region level": region_to_code.keys(), "shared code": region_to_code.values()}
)

print("Numerical parameters learned from X and Y together")
display(numeric_parameters)
print("Categorical codebook learned from X and Y together")
display(region_codebook)

Numerical parameters learned from X and Y together


,median,minimum,maximum,range
age,39.000,24.000,60.000,36.000
income_k,55.000,32.000,91.000,59.000


Categorical codebook learned from X and Y together


,region level,shared code
0,East,0
1,North,1
2,South,2
3,West,3
4,__MISSING__,4


For a numerical column $j$, let $a_j$ and $b_j$ be its pooled minimum and maximum and let $q_j$ be its pooled median. Define

$$
z_{ij}^{*}=
\begin{cases}
z_{ij}, & z_{ij}\text{ is observed},\\
q_j, & z_{ij}\text{ is missing},
\end{cases}
\qquad
\widetilde z_{ij}=\frac{z_{ij}^{*}-a_j}{b_j-a_j}.
$$

The companion indicator $m_{ij}=\mathbf 1\{z_{ij}=\mathrm{NA}\}$ prevents a filled value from being mistaken for an originally observed median. A categorical value is replaced by its shared code, with `__MISSING__` treated as an explicit level.

In [7]:
#| label: apply-shared-transformation

def transform_to_metric_view(frame):
    transformed = pd.DataFrame(index=frame.index)

    for column in NUMERIC_COLUMNS:
        parameters = numeric_parameters.loc[column]
        missing = frame[column].isna()
        filled = frame[column].astype("Float64").fillna(parameters["median"])
        transformed[column] = (
            (filled.astype(float) - parameters["minimum"]) / parameters["range"]
        )
        transformed[f"{column}__missing"] = missing.astype("int8")

    categorical = frame[CATEGORICAL_COLUMN].fillna(MISSING_LEVEL)
    transformed[f"{CATEGORICAL_COLUMN}__code"] = (
        categorical.map(region_to_code).astype("int64")
    )
    transformed["outcome"] = frame["outcome"].astype("int64")
    return transformed


X_tilde = transform_to_metric_view(X)
Y_tilde = transform_to_metric_view(Y)

print("X_tilde: the complete metric-ready representation of X")
display(X_tilde)
print("Y_tilde: the complete metric-ready representation of Y")
display(Y_tilde)

X_tilde: the complete metric-ready representation of X


,age,age__missing,income_k,income_k__missing,region__code,outcome
x1,0.000,0,0.000,0,1,0
x2,0.194,0,0.271,0,2,0
x3,0.417,1,0.390,0,1,1
x4,0.611,0,0.390,1,3,1
x5,0.944,0,1.000,0,4,1
x6,0.417,0,0.593,0,2,0


Y_tilde: the complete metric-ready representation of Y


,age,age__missing,income_k,income_k__missing,region__code,outcome
y1,0.028,0,0.051,0,1,0
y2,0.417,1,0.322,0,2,1
y3,0.556,0,0.390,1,3,1
y4,1.000,0,0.949,0,4,1
y5,0.361,0,0.542,0,0,0


In [8]:
#| label: verify-complete-representations

assert X_tilde.columns.equals(Y_tilde.columns)
assert not X_tilde.isna().any().any()
assert not Y_tilde.isna().any().any()

print("Same transformed columns:", X_tilde.columns.equals(Y_tilde.columns))
print("Missing cells in X_tilde:", int(X_tilde.isna().sum().sum()))
print("Missing cells in Y_tilde:", int(Y_tilde.isna().sum().sum()))

Same transformed columns: True
Missing cells in X_tilde: 0
Missing cells in Y_tilde: 0


> **Important:** $\widetilde X$ and $\widetilde Y$ are metric-ready views, not claims about the unknown true values. Median filling here makes a numerical representation complete; the indicator columns and explicit missing category retain information about where values were absent.

## 4. Follow individual cells through the transformation

The notation becomes concrete when we trace cells from the original tables into their masks and complete representations.

In [9]:
#| label: trace-numerical-cell

age_parameters = numeric_parameters.loc["age"]
filled_age = age_parameters["median"]
scaled_age = (filled_age - age_parameters["minimum"]) / age_parameters["range"]

print("Trace X[x3, age]")
print("  original value:       ", X.loc["x3", "age"])
print("  M_X[x3, age]:        ", M_X.loc["x3", "age"])
print(f"  pooled median used:   {filled_age:.0f}")
print(f"  scaled median:        ({filled_age:.0f} - {age_parameters['minimum']:.0f}) / "
      f"({age_parameters['maximum']:.0f} - {age_parameters['minimum']:.0f}) = {scaled_age:.3f}")
print("  X_tilde[x3, age]:    ", round(X_tilde.loc["x3", "age"], 3))
print("  missingness indicator:", X_tilde.loc["x3", "age__missing"])

Trace X[x3, age]
  original value:        <NA>
  M_X[x3, age]:         1
  pooled median used:   39
  scaled median:        (39 - 24) / (60 - 24) = 0.417
  X_tilde[x3, age]:     0.417
  missingness indicator: 1


In [10]:
#| label: trace-categorical-cells

categorical_trace = pd.DataFrame(
    [
        {
            "cell": "X[x5, region]",
            "original": X.loc["x5", "region"],
            "mask": M_X.loc["x5", "region"],
            "level used": MISSING_LEVEL,
            "shared code": X_tilde.loc["x5", "region__code"],
        },
        {
            "cell": "Y[y5, region]",
            "original": Y.loc["y5", "region"],
            "mask": M_Y.loc["y5", "region"],
            "level used": Y.loc["y5", "region"],
            "shared code": Y_tilde.loc["y5", "region__code"],
        },
    ]
)
display(categorical_trace)

,cell,original,mask,level used,shared code
0,"X[x5, region]",<NA>,1,__MISSING__,4
1,"Y[y5, region]",East,0,East,0


The `East` level appears only in $Y$, yet it belongs to the shared codebook because that codebook was fitted on $X\cup Y$. The missing region in $X$ receives the explicit missing code rather than being confused with `East`, `North`, `South`, or `West`.

## 5. The two metric branches

The series will build individual metrics later. For now, the important split is the input view each family needs.

| Input view | Notation | What it preserves | Metric families introduced later |
|---|---|---|---|
| Original tables | $X,Y$ | Original values and actual missing cells | Missingness fidelity |
| Missingness masks | $M_X,M_Y$ | Locations and row-wise patterns of missing values | Marginal and joint missingness |
| Complete metric-ready tables | $\widetilde X,\widetilde Y$ | Aligned numerical values plus explicit missingness information | Value fidelity, utility, and privacy |

The relationship can be summarized as

$$
(X,Y)
\longrightarrow
\begin{cases}
(M_X,M_Y), & \text{missingness metrics},\\[4pt]
(\widetilde X,\widetilde Y)=
\bigl(T_{X\cup Y}(X),T_{X\cup Y}(Y)\bigr),
& \text{value, utility, and privacy metrics}.
\end{cases}
$$

When an independent holdout is needed later, it will be denoted by $H$ and transformed with the already fitted rules: $\widetilde H=T_{X\cup Y}(H)$. The holdout is not used to fit the transformation.

### Complete versus incomplete original tables

- If $X$ and $Y$ are already complete, then $M_X$ and $M_Y$ contain only zeros. The complete-representation metrics remain applicable, while missingness metrics carry no additional information.
- If either table is incomplete, both branches matter: the original tables reveal the missingness process, while $\widetilde X$ and $\widetilde Y$ provide aligned complete inputs for the metrics that require them.

This is why we do not introduce separate symbols such as $X_c$ and $X_n$. One table can have both an original incomplete view and a complete metric-ready view.

## Check your understanding

1. Why is $(M_X)_{x3,\,age}=1$ even though the corresponding value in $\widetilde X$ is numerical?
2. What could go wrong if `region` were encoded separately in $X$ and $Y$?
3. If both original tables were complete, which objects would still be useful?
4. Why is $\widetilde X$ not described as the true completed version of $X$?
5. If a future metric uses $H$, why must $H$ remain outside the fitting of $T_{X\cup Y}$?

## Takeaway

We now have a precise vocabulary for the entire series:

- $X,Y$: the original same-schema tables;
- $M_X,M_Y$: binary maps of their missing cells;
- $T_{X\cup Y}$: one transformation fitted symmetrically to the compared pair; and
- $\widetilde X,\widetilde Y$: aligned, complete representations used only when a metric needs them.

The next lessons will introduce the comparison metrics one at a time, beginning with their mathematical definitions and then implementing them on small visible examples.